# 🎼 Chord Analysis Plots

**Difficulty:** 🟡 Medium  
**Prerequisites:** [Piano Rolls](01_piano_rolls.ipynb), [Extracting Chords](../02_intermediate/03_extracting_chords.ipynb)  
**Time:** 30-35 minutes  

---

## What You'll Learn

- How to visualize chord complexity over time
- How to use `plotChordsNumberOfNotes()` for harmonic density
- How to create custom chord analysis visualizations
- How to track chord changes and harmonic rhythm
- How to identify consonance vs. dissonance patterns
- How to compare harmonic complexity across pieces

## Why This Matters

Harmony is the vertical dimension of music - what notes sound together. Understanding harmonic complexity helps you:

- **Identify texture changes**: Dense chords vs. sparse harmonies
- **Track harmonic rhythm**: How often do chords change?
- **Analyze orchestration**: How many notes at each moment?
- **Study compositional techniques**: Buildup to climaxes, contrast, development
- **Compare styles**: Bach vs. Wagner vs. Stravinsky harmonic approaches

This is essential for:
- Music theorists analyzing harmonic progressions
- Composers learning orchestration techniques
- Researchers studying harmonic complexity
- Teachers demonstrating texture and harmony concepts

---

## Step 1: Setup and Import

In [ ]:
# Import maialib and other libraries
import maialib as ml
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"✅ Maialib {ml.__version__} loaded successfully!")
print("🎼 Ready to analyze harmony!")

---

## Step 2: Basic Chord Density Visualization

Let's start by visualizing how many notes sound together at each moment - a measure of harmonic density.

In [ ]:
# Load Bach Prelude No. 1 (piano piece with clear harmonies)
bach = ml.Score('Bach_Prelude_1_BWV_846.mxl')

print("Score Information:")
print(f"  Title: {bach.getWorkTitle()}")
print(f"  Composer: {bach.getComposer()}")
print(f"  Measures: {bach.getNumMeasures()}")

# Extract chords from the score
chords = bach.getChords()

print(f"\nExtracted {len(chords)} chord moments")
print(f"First chord: {chords[0].getName()} with {chords[0].size()} notes")

# Visualize chord density
ml.plotChordsNumberOfNotes(bach)

print("\n📊 Chord density plot created!")

**What You're Seeing:**

- **X-axis**: Time (measure number)
- **Y-axis**: Number of simultaneously sounding notes
- **Line plot**: Shows harmonic density evolution

**Musical Insight**: In the Bach Prelude:
- Notice the **remarkably consistent** number of notes (usually 5-6)
- This creates the characteristic **flowing texture**
- Moments where density changes often mark **structural boundaries**
- The **regularity** is what makes this piece so hypnotic

**Historical Context**: Bach wrote this as a teaching piece, demonstrating how to sustain interest with minimal textural variation!

---

## Step 3: Analyzing Harmonic Rhythm

Harmonic rhythm = how often the harmony changes. Let's visualize chord changes over time.

In [ ]:
# Load a score
score = ml.Score('Mozart_Requiem_Introitus.mxl')

# Extract chords
chords = score.getChords()

# Create a list to track chord changes
chord_data = []

for i, chord in enumerate(chords):
    chord_data.append({
        'index': i,
        'name': chord.getName(),
        'num_notes': chord.size(),
        'root': chord.getRoot().getPitchClass() if chord.size() > 0 else 'Rest',
    })

# Convert to DataFrame
df_chords = pd.DataFrame(chord_data)

# Count chord changes
chord_changes = (df_chords['name'] != df_chords['name'].shift()).sum()

print(f"Total chord moments: {len(chords)}")
print(f"Chord changes: {chord_changes}")
print(f"Average duration per chord: {len(chords) / chord_changes:.2f} moments")

# Visualize chord density
ml.plotChordsNumberOfNotes(score)

print("\n📊 Harmonic rhythm visualization created!")
print("\nFirst 10 chords:")
print(df_chords[['index', 'name', 'num_notes']].head(10))

**Understanding Harmonic Rhythm:**

**Fast harmonic rhythm** (frequent chord changes):
- Creates momentum and forward drive
- Common in development sections
- Can create instability or tension
- Examples: Bach fugues, Vivaldi sequences

**Slow harmonic rhythm** (infrequent changes):
- Creates stability and calm
- Common in introductions and codas
- Allows focus on melody or rhythm
- Examples: Chopin nocturnes, Wagner preludes

**Varied harmonic rhythm**:
- Creates interest through contrast
- Marks structural boundaries
- Typical of sonata form and other complex forms

---

## Step 4: Custom Visualization - Chord Complexity Heat Map

Let's create a custom visualization showing both chord density AND chord changes.

In [ ]:
import plotly.express as px

def plot_chord_complexity_heatmap(score, window_size=4):
    """
    Create a heatmap showing chord complexity over time.
    
    Args:
        score: Maialib Score object
        window_size: Number of measures per window (for aggregation)
    """
    
    # Extract chords
    chords = score.getChords()
    
    # Get measure information for each chord
    df = score.toDataFrame()
    
    # Calculate chord density by measure
    measures = df.groupby('measure').agg({
        'pitch': 'count',  # Number of notes
        'midi': ['min', 'max', 'std']  # Pitch statistics
    }).reset_index()
    
    measures.columns = ['measure', 'note_count', 'min_pitch', 'max_pitch', 'pitch_std']
    
    # Calculate complexity metrics
    measures['pitch_range'] = measures['max_pitch'] - measures['min_pitch']
    measures['complexity'] = measures['note_count'] * (1 + measures['pitch_std'].fillna(0) / 10)
    
    # Group into windows
    measures['window'] = measures['measure'] // window_size
    
    # Aggregate by window
    windows = measures.groupby('window').agg({
        'note_count': 'mean',
        'pitch_range': 'mean',
        'complexity': 'mean'
    }).reset_index()
    
    # Create heatmap data
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=('Average Note Count', 'Average Pitch Range', 'Harmonic Complexity'),
        vertical_spacing=0.1
    )
    
    # Note count
    fig.add_trace(
        go.Bar(x=windows['window']*window_size, y=windows['note_count'], 
               name='Note Count', marker_color='steelblue'),
        row=1, col=1
    )
    
    # Pitch range
    fig.add_trace(
        go.Bar(x=windows['window']*window_size, y=windows['pitch_range'], 
               name='Pitch Range', marker_color='orange'),
        row=2, col=1
    )
    
    # Complexity
    fig.add_trace(
        go.Bar(x=windows['window']*window_size, y=windows['complexity'], 
               name='Complexity', marker_color='crimson'),
        row=3, col=1
    )
    
    # Update layout
    fig.update_xaxes(title_text="Measure", row=3, col=1)
    fig.update_layout(
        height=800,
        title_text=f"Harmonic Complexity Analysis: {score.getWorkTitle()}",
        showlegend=False
    )
    
    fig.show()

# Test it
beethoven = ml.Score('Beethoven_Symphony_5_mov_1.xml')

print("Creating chord complexity heatmap...")
plot_chord_complexity_heatmap(beethoven, window_size=4)

**Interpreting the Heatmap:**

**Note Count (Top panel)**:
- High values: Dense orchestration, tutti passages
- Low values: Solo passages, sparse texture
- Changes: Indicate orchestration shifts

**Pitch Range (Middle panel)**:
- High values: Wide spacing between parts (dramatic)
- Low values: Narrow spacing (blended, homogeneous)
- Related to register and voicing choices

**Complexity (Bottom panel)**:
- Combines density + pitch variety
- High values: Complex, thick texture
- Low values: Simple, clear texture
- Useful for identifying climaxes and structural boundaries

**In Beethoven's Symphony No. 5**, notice:
- The famous opening has HIGH complexity (full orchestra, wide range)
- Transitional passages have LOWER complexity
- Development section shows VARIED complexity (constant changes)

---

## Step 5: Tracking Chord Types Over Time

Let's analyze what kinds of chords are used and when.

In [ ]:
def analyze_chord_types(score):
    """
    Analyze the types of chords used in a score.
    """
    
    chords = score.getChords()
    
    # Categorize chords
    chord_stats = {
        'major': 0,
        'minor': 0,
        'diminished': 0,
        'augmented': 0,
        'dominant7': 0,
        'other': 0,
        'dyads': 0,  # 2 notes
        'triads': 0,  # 3 notes
        'seventh_chords': 0,  # 4 notes
        'extended': 0,  # 5+ notes
    }
    
    for chord in chords:
        size = chord.size()
        
        # Skip empty chords (rests)
        if size == 0:
            continue
        
        # Categorize by quality
        if chord.isMajor():
            chord_stats['major'] += 1
        elif chord.isMinor():
            chord_stats['minor'] += 1
        elif chord.isDiminished():
            chord_stats['diminished'] += 1
        elif chord.isAugmented():
            chord_stats['augmented'] += 1
        elif chord.isDominantSeventh():
            chord_stats['dominant7'] += 1
        else:
            chord_stats['other'] += 1
        
        # Categorize by size
        if size == 2:
            chord_stats['dyads'] += 1
        elif size == 3:
            chord_stats['triads'] += 1
        elif size == 4:
            chord_stats['seventh_chords'] += 1
        elif size >= 5:
            chord_stats['extended'] += 1
    
    return chord_stats

# Analyze Bach
bach = ml.Score('Bach_Prelude_1_BWV_846.mxl')
bach_stats = analyze_chord_types(bach)

print("Bach Prelude No. 1 - Chord Types:")
print("\nBy Quality:")
print(f"  Major chords: {bach_stats['major']}")
print(f"  Minor chords: {bach_stats['minor']}")
print(f"  Diminished: {bach_stats['diminished']}")
print(f"  Augmented: {bach_stats['augmented']}")
print(f"  Dominant 7th: {bach_stats['dominant7']}")
print(f"  Other: {bach_stats['other']}")

print("\nBy Size:")
print(f"  Dyads (2 notes): {bach_stats['dyads']}")
print(f"  Triads (3 notes): {bach_stats['triads']}")
print(f"  Seventh chords (4 notes): {bach_stats['seventh_chords']}")
print(f"  Extended (5+ notes): {bach_stats['extended']}")

# Create a pie chart
quality_data = {
    'Major': bach_stats['major'],
    'Minor': bach_stats['minor'],
    'Diminished': bach_stats['diminished'],
    'Dominant 7th': bach_stats['dominant7'],
    'Other': bach_stats['other']
}

fig = go.Figure(data=[go.Pie(
    labels=list(quality_data.keys()),
    values=list(quality_data.values()),
    hole=0.3
)])

fig.update_layout(
    title="Chord Quality Distribution: Bach Prelude No. 1",
    height=500
)

fig.show()

**Chord Type Analysis Insights:**

**For Bach Prelude No. 1 (in C Major)**:
- Expect **majority major chords** (piece is in a major key)
- Some **minor chords** (relative minor, subdominant minor)
- **Diminished chords** for tension and voice leading
- **Dominant 7th chords** creating strong resolutions
- Mostly **7th chords and extended** (Bach's rich harmony)

**Comparative Harmony:**
- **Baroque** (Bach): Functional harmony, clear progressions, lots of 7th chords
- **Classical** (Mozart, Haydn): Cleaner triads, less chromaticism
- **Romantic** (Chopin, Wagner): More chromatic, diminished 7ths, augmented chords
- **20th Century** (Stravinsky, Debussy): Clusters, quartal harmony, extended chords

---

## Step 6: Comparing Harmonic Complexity Across Composers

Let's compare how different composers use harmony.

In [ ]:
# Load multiple scores
scores = {
    'Bach': ml.Score('Bach_Prelude_1_BWV_846.mxl'),
    'Mozart': ml.Score('Mozart_Requiem_Introitus.mxl'),
    'Beethoven': ml.Score('Beethoven_Symphony_5_mov_1.xml'),
    'Chopin': ml.Score('Chopin_Fantasie_Impromptu.mxl'),
}

# Analyze each
comparison_data = []

for name, score in scores.items():
    chords = score.getChords()
    
    # Calculate average chord size
    chord_sizes = [c.size() for c in chords if c.size() > 0]
    avg_size = sum(chord_sizes) / len(chord_sizes) if chord_sizes else 0
    
    # Count chord changes
    chord_names = [c.getName() for c in chords]
    changes = sum(1 for i in range(1, len(chord_names)) if chord_names[i] != chord_names[i-1])
    change_rate = changes / len(chords) if chords else 0
    
    # Get pitch statistics
    df = score.toDataFrame()
    pitch_std = df.groupby('measure')['midi'].std().mean()
    
    comparison_data.append({
        'Composer': name,
        'Avg Chord Size': avg_size,
        'Chord Change Rate': change_rate,
        'Pitch Variety': pitch_std
    })

# Create comparison DataFrame
df_comparison = pd.DataFrame(comparison_data)

print("Harmonic Complexity Comparison:")
print(df_comparison.to_string(index=False))

# Visualize comparison
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Average Chord Size', 'Chord Change Rate', 'Pitch Variety')
)

# Chord size
fig.add_trace(
    go.Bar(x=df_comparison['Composer'], y=df_comparison['Avg Chord Size'], 
           name='Chord Size', marker_color='steelblue'),
    row=1, col=1
)

# Change rate
fig.add_trace(
    go.Bar(x=df_comparison['Composer'], y=df_comparison['Chord Change Rate'], 
           name='Change Rate', marker_color='orange'),
    row=1, col=2
)

# Pitch variety
fig.add_trace(
    go.Bar(x=df_comparison['Composer'], y=df_comparison['Pitch Variety'], 
           name='Pitch Variety', marker_color='crimson'),
    row=1, col=3
)

fig.update_layout(
    height=400,
    title_text="Comparative Harmonic Analysis",
    showlegend=False
)

fig.show()

**Understanding the Comparison:**

**Average Chord Size**:
- Reflects orchestration and voicing
- Piano music (Bach, Chopin): Moderate (4-6 notes)
- Orchestral music: Can be very high (10+ notes with doubling)

**Chord Change Rate**:
- Harmonic rhythm speed
- High: Fast-moving, unstable (development sections)
- Low: Stable, pedal points, extended harmonies

**Pitch Variety**:
- How much pitches vary within measures
- High: Wide spacing, big leaps, dramatic
- Low: Close spacing, stepwise motion, lyrical

**Expected Patterns**:
- **Bach**: High complexity, functional harmony, fast changes
- **Mozart**: Moderate complexity, elegant simplicity
- **Beethoven**: Variable complexity, dramatic contrasts
- **Chopin**: Dense piano textures, rich chromaticism

---

## 🎯 Practice Exercises

### Exercise 1: Analyze Your Own Score

Choose any score and:
1. Create a chord density plot
2. Calculate average chord size
3. Identify the most complex moment (highest density)

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
# Load score
score = ml.Score('Strauss_Blue_Danube_Waltz.mxl')

# Get chords
chords = score.getChords()

# Calculate stats
chord_sizes = [c.size() for c in chords if c.size() > 0]
avg_size = sum(chord_sizes) / len(chord_sizes)
max_size = max(chord_sizes)
max_idx = chord_sizes.index(max_size)

print(f"Average chord size: {avg_size:.2f}")
print(f"Maximum chord size: {max_size} notes at chord {max_idx}")

# Visualize
ml.plotChordsNumberOfNotes(score)
```
</details>

---

### Exercise 2: Compare Two Pieces

Compare harmonic complexity between two contrasting pieces (e.g., Bach vs. Chopin).

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
# Load both scores
bach = ml.Score('Bach_Cello_Suite_1.mxl')
chopin = ml.Score('Chopin_Fantasie_Impromptu.mxl')

# Analyze Bach
bach_chords = bach.getChords()
bach_sizes = [c.size() for c in bach_chords if c.size() > 0]
bach_avg = sum(bach_sizes) / len(bach_sizes)

# Analyze Chopin
chopin_chords = chopin.getChords()
chopin_sizes = [c.size() for c in chopin_chords if c.size() > 0]
chopin_avg = sum(chopin_sizes) / len(chopin_sizes)

print(f"Bach average chord size: {bach_avg:.2f}")
print(f"Chopin average chord size: {chopin_avg:.2f}")

# Visualize both
print("\nBach:")
ml.plotChordsNumberOfNotes(bach)

print("\nChopin:")
ml.plotChordsNumberOfNotes(chopin)
```
</details>

---

### Exercise 3: Find Chord Type Distribution

Use the `analyze_chord_types()` function to compare major vs. minor chord usage in different pieces.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
# Compare major-key piece vs. minor-key piece
major_piece = ml.Score('Bach_Prelude_1_BWV_846.mxl')  # C Major
minor_piece = ml.Score('Chopin_Fantasie_Impromptu.mxl')  # C# Minor

# Analyze both
major_stats = analyze_chord_types(major_piece)
minor_stats = analyze_chord_types(minor_piece)

# Calculate ratios
major_ratio_major = major_stats['major'] / (major_stats['major'] + major_stats['minor']) if (major_stats['major'] + major_stats['minor']) > 0 else 0
minor_ratio_major = minor_stats['major'] / (minor_stats['major'] + minor_stats['minor']) if (minor_stats['major'] + minor_stats['minor']) > 0 else 0

print(f"Bach (C Major): {major_ratio_major*100:.1f}% major chords")
print(f"Chopin (C# Minor): {minor_ratio_major*100:.1f}% major chords")
```
</details>

---

## ✅ Summary

Congratulations! You can now visualize and analyze harmonic complexity. You've learned:

- ✅ How to use `plotChordsNumberOfNotes()` for density visualization
- ✅ How to analyze harmonic rhythm and chord changes
- ✅ How to create custom chord complexity visualizations
- ✅ How to categorize and count chord types
- ✅ How to compare harmonic approaches across composers
- ✅ How to quantify texture and orchestration

## Key Concepts

- **Harmonic density** = number of simultaneously sounding notes
- **Harmonic rhythm** = rate of chord changes
- **Chord complexity** = combination of density + variety
- **Texture analysis** reveals orchestration and compositional choices
- **Different eras have different harmonic signatures**

## 🎯 Next Steps

Continue exploring visualization and analysis:

1. **[Parts Activity](04_parts_activity.ipynb)** - See orchestration patterns
2. **[Combining Visualizations](06_combining_visualizations.ipynb)** - Multi-panel analysis
3. **[Harmonic Complexity Analysis](../05_analysis/01_harmonic_complexity_analysis.ipynb)** - Deep dive into harmony
4. **[Dissonance Curves](../05_analysis/02_dissonance_curves_analysis.ipynb)** - Psychoacoustic analysis

---

<div align="center">

**See harmony evolve! 🎼**

*Chord visualizations reveal the vertical dimension of music - how composers create tension, release, and emotional journey*

</div>